BUILDING AGENTIC RAG PIPLINE

In [3]:
from dotenv import load_dotenv
load_dotenv()

True

In [5]:
from langchain_community.document_loaders  import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_google_genai import GoogleGenerativeAIEmbeddings , ChatGoogleGenerativeAI
from langchain_community.vectorstores import InMemoryVectorStore


In [12]:
loader = PyPDFLoader("../apps/Data/Patientreport.pdf")
docs=loader.load()

fontTools is required to fully parse the encoding of a CFF Type1 font in font dictionary {'/FirstChar': 32, '/FontDescriptor': IndirectObject(565, 0, 2415442049808), '/Type': '/Font', '/LastChar': 114, '/Subtype': '/Type1', '/BaseFont': '/MYIBIJ+Poppins-Regular', '/Encoding': '/WinAnsiEncoding', '/Widths': [267, 0, 0, 0, 0, 0, 739, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 620, 0, 0, 0, 0, 0, 0, 0, 1030, 0, 640, 0, 0, 373]}, but is not installed. Consider installing fontTools if you encounter encoding problems.


In [13]:
splitter=RecursiveCharacterTextSplitter(chunk_size=1000,chunk_overlap=200)
splitted_docs=splitter.split_documents(docs)

In [24]:
embbedings = GoogleGenerativeAIEmbeddings(model="gemini-embedding-2-preview")
from langchain_chroma import Chroma
vector_store=Chroma(
    
   embedding_function=embbedings, 
   persist_directory="./agentic_Rag.db"

)
vector_store.add_documents(splitted_docs)


['fef3b4f9-0ca8-4b32-80cb-c091f57d0a6e',
 'db6b8ceb-e53a-4820-b983-ea0f790d6f05',
 'ccc6736b-958b-4916-86ae-7667c54d449a',
 '364bb499-a687-43b5-952c-b2ea69d4d5ce',
 '6e7efd70-660e-41d1-98e6-8d80bbd7d3cb',
 'cec4b2c1-e4b1-47d9-a7d5-3ed0d0c4bed5',
 'a48478f6-9626-4595-8b6c-16242153e2bb',
 'ace02560-d880-4404-8492-52f030b563b7',
 'f8f772bb-8790-4244-885f-518cde48b2f8',
 '8f59d4d2-3b6d-499b-a323-d9f0f63d0434',
 '8a2c6bc2-2be9-4abd-8546-08e681f99ac3',
 '0b05654f-32b5-4445-878a-931054be8371',
 'd6419308-ef15-4f38-ad41-a58d96153136',
 '15270152-1196-46f3-ad5a-385bd367f13c',
 'f4644654-9fab-4610-89eb-5e57fdfe4a05',
 '76588226-0d7f-46f8-a000-78ae098fcbd1',
 '15137ab9-8aec-4ee9-974c-c69c72e9d5dd',
 '4841c57c-8052-418e-bd0a-7c99e79ce2aa',
 '72094397-bb54-493e-a57c-8470cde3aec4',
 '9cdd8583-e43f-4659-b394-f969a42dee3c',
 '42e42286-74d3-49cb-9089-8fb7bab86dcf',
 '47e2059b-1e40-41c2-b944-a878fff2f783',
 'e4be2d5c-68db-4b5c-927d-69d3960fed92',
 '5f95469c-a63a-4f7b-9ff7-eed7657eb7af',
 'cf298d6e-5584-

Building ai agent 


In [ ]:
#for building agent we need tool ,LLM model prompt
from langchain.tools import tool
@tool
def retriver_tool(query:str):
    ''' 
     This tool can help you to retrive the relevant data of pdf documents and these pdfs document have details about medical
    '''
    docs= vector_store.similarity_search(query=query , k=4)
    print("tool used" , query)
    context=""
    for doc in docs:
        context+=doc.page_content + "\n\n"
    return context

    

In [30]:
llm  =ChatGoogleGenerativeAI(model="gemini-3.7-flash")
prompt = """   you are a helpful assistant answer questions using retrieved context ALWAYS use retrieval tool for questions requiring external knowledege  """
from langchain.agents import create_agent

agent=create_agent(model=llm , tools=[retriver_tool] , system_prompt=prompt) 

In [ ]:
query="what is the name of patient and name of doctors"
response=agent.invoke({"messages":[{"role":"user" , "content":query}]})
response["messages"][-1].content

AttributeError: 'Chroma' object has no attribute 'similartiy_search'